In [ ]:
!pip install openai pandas python-dotenv

In [ ]:
import os
import json
import pandas as pd
import openai
import time
from google.colab import userdata

In [ ]:
# Securely load your API key
openai.api_key = userdata.get('OPENAI_API_KEY')

In [ ]:
class DocumentAnalyzer:
    """A class for analyzing documents using OpenAI's API."""

    def __init__(self, api_key=None, model="gpt-3.5-turbo", max_chars=4000):
        """Initialize the DocumentAnalyzer with configuration."""
        self.client = openai.OpenAI(api_key=api_key or userdata.get('OPENAI_API_KEY'))
        self.model = model
        self.max_chars = max_chars
        self.analysis_history = []

    def read_text_file(self, file_path):
        """Read content from a text file."""
        try:
            with open(file_path, 'r', encoding='utf-8') as file:
                return file.read()
        except Exception as e:
            print(f"Error reading file: {e}")
            return None

    def extract_entities(self, text):
        """Extract key entities from text: people, organizations, locations, dates."""
        if not text:
            return None

        prompt = f"""
        Extract the following entities from the text below:
        1. People (names of individuals)
        2. Organizations (companies, institutions, etc.)
        3. Locations (cities, countries, etc.)
        4. Dates (any date references)

        Return the results as a JSON object with these categories as keys and lists as values.

        Text:
        {text[:self.max_chars]}
        """

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are an expert at extracting structured information from text."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.0,
                response_format={"type": "json_object"}
            )

            result = json.loads(response.choices[0].message.content)
            return result

        except Exception as e:
            print(f"Error in entity extraction: {e}")
            return None

    def generate_summary(self, text):
        """Generate a concise summary of the document content."""
        if not text:
            return None

        prompt = f"""
        Please provide a concise summary of the following text, highlighting the main points and key information.

        Text:
        {text[:self.max_chars]}
        """

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are an expert at summarizing documents concisely."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.3
            )

            return response.choices[0].message.content

        except Exception as e:
            print(f"Error in summary generation: {e}")
            return None

    def classify_document(self, text):
        """Classify document type, formality, and purpose."""
        if not text:
            return None

        prompt = f"""
        Classify the following document on these dimensions:
        1. Type: technical or business
        2. Formality: formal or informal
        3. Purpose: contract, report, proposal, email, or other

        Return as JSON with keys: type, formality, purpose

        Text:
        {text[:self.max_chars]}
        """

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are an expert at document classification."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.0,
                response_format={"type": "json_object"}
            )

            result = json.loads(response.choices[0].message.content)
            return result

        except Exception as e:
            print(f"Error in document classification: {e}")
            return None

    def analyze_document(self, file_path):
        """Analyze a document by extracting entities, generating summary, and classifying."""
        print(f"Analyzing document: {file_path}")

        # Read the document
        text = self.read_text_file(file_path)
        if not text:
            return None

        print(f"Document loaded, length: {len(text)} characters")

        # Perform analysis
        print("Extracting entities...")
        entities = self.extract_entities(text)

        print("Generating summary...")
        summary = self.generate_summary(text)

        print("Classifying document...")
        classification = self.classify_document(text)

        # Combine results
        analysis = {
            "filename": os.path.basename(file_path),
            "character_count": len(text),
            "summary": summary,
            "entities": entities,
            "classification": classification,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
        }

        # Store in history
        self.analysis_history.append(analysis)

        return analysis

    def display_analysis(self, analysis):
        """Display the document analysis in a structured format."""
        if not analysis:
            print("No analysis data available.")
            return

        print("\n" + "="*50)
        print(f"DOCUMENT ANALYSIS: {analysis['filename']}")
        print("="*50)

        print("\nSUMMARY:")
        print("-"*50)
        print(analysis['summary'])
        print("-"*50)

        print("\nCLASSIFICATION:")
        print("-"*50)
        if analysis['classification']:
            for key, value in analysis['classification'].items():
                print(f"  {key.capitalize()}: {value}")
        print("-"*50)

        print("\nEXTRACTED ENTITIES:")
        print("-"*50)
        if analysis['entities']:
            for entity_type, entities in analysis['entities'].items():
                if entities:
                    print(f"\n{entity_type.upper()}:")
                    for entity in entities:
                        print(f"  - {entity}")

        print("-"*50)
        print(f"Character count: {analysis['character_count']}")
        print(f"Analyzed on: {analysis['timestamp']}")
        print("="*50 + "\n")

    def save_analysis(self, analysis, filename="analysis_result.json"):
        """Save analysis results to a JSON file."""
        try:
            with open(filename, "w", encoding="utf-8") as f:
                json.dump(analysis, f, indent=2)
            print(f"Analysis saved to {filename}")
        except Exception as e:
            print(f"Error saving analysis: {e}")

    def get_analysis_history(self):
        """Return the history of all analyses performed."""
        return self.analysis_history

# Usage example
def run_example_with_class():
    """Run an example analysis using the DocumentAnalyzer class."""
    # Create sample text
    sample_text = """
    Document Analysis Example:
    John Smith, CEO of TechCorp, announced on January 15, 2023 that the company
    will open a new office in Seattle, Washington. The expansion is part of TechCorp's
    growth strategy for North America. Sarah Johnson, the VP of Operations, will lead
    the new office starting March 1, 2023. The company, founded in 2010, has been working
    with Microsoft and Amazon on several AI projects in the past year.
    """

    # Save sample text to a file
    with open("sample_document.txt", "w", encoding="utf-8") as f:
        f.write(sample_text)

    # Create analyzer instance
    analyzer = DocumentAnalyzer()

    # Analyze the document
    analysis = analyzer.analyze_document("sample_document.txt")

    # Display and save results
    if analysis:
        analyzer.display_analysis(analysis)
        analyzer.save_analysis(analysis)

        # Show analysis history
        print(f"Total analyses performed: {len(analyzer.get_analysis_history())}")

# Run the example
run_example_with_class()

Analyzing document: sample_document.txt
Document loaded, length: 450 characters
Extracting entities...
Generating summary...
Classifying document...

DOCUMENT ANALYSIS: sample_document.txt

SUMMARY:
--------------------------------------------------
Summary:
TechCorp's CEO, John Smith, revealed plans to establish a new office in Seattle, Washington on January 15, 2023, as part of the company's North American growth strategy. Sarah Johnson, the VP of Operations, will head the new office from March 1, 2023. Founded in 2010, TechCorp has collaborated with Microsoft and Amazon on various AI projects in the last year.
--------------------------------------------------

CLASSIFICATION:
--------------------------------------------------
  Type: business
  Formality: formal
  Purpose: announcement
--------------------------------------------------

EXTRACTED ENTITIES:
--------------------------------------------------

PEOPLE:
  - John Smith
  - Sarah Johnson

ORGANIZATIONS:
  - TechCorp
  - M